In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils import *


In [ ]:
root_dir, data_dir = get_notebook_paths()
MIMIC_CXR_JPG_DIR = data_dir / "MIMIC-CXR-JPG"
DATA_CAMCHEX = data_dir / "camchex"

print(Path.cwd())
print(root_dir)
print(data_dir)


In [ ]:
train = pd.read_csv(DATA_CAMCHEX / "03_mimic_train.csv")
test = pd.read_csv(DATA_CAMCHEX / "03_mimic_test.csv")
dev = pd.read_csv(DATA_CAMCHEX / "03_mimic_development.csv")

train["split"] = "train"
test["split"] = "test"
dev["split"] = "dev"

data = pd.concat([train, test, dev], axis=0).reset_index(drop=True)
print(data.shape)
data.head()

del train, test, dev


In [ ]:
LABEL_COLS = [
    "Atelectasis", "Calcification of the Aorta", "Cardiomegaly", "Consolidation",
    "Edema", "Emphysema", "Enlarged Cardiomediastinum", "Fibrosis", "Fracture",
    "Hernia", "Infiltration", "Lung Lesion", "Lung Opacity", "Mass", "No Finding",
    "Nodule", "Pleural Effusion", "Pleural Other", "Pleural Thickening",
    "Pneumomediastinum", "Pneumonia", "Pneumoperitoneum", "Pneumothorax",
    "Subcutaneous Emphysema", "Support Devices", "Tortuous Aorta",
]

VITAL_COLS = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp"]


In [ ]:
# Split summary

split_summary = (
    data.groupby("split")
    .agg(
        rows=("dicom_id", "count"),
        patients=("subject_id", "nunique"),
        studies=("study_id", "nunique"),
        images=("dicom_id", "nunique"),
    )
    .reset_index()
)

split_summary


In [ ]:
# Check patient overlap

patients = {
    split: set(data.loc[data["split"] == split, "subject_id"])
    for split in ["train", "val", "test"]
}

for a in ["train", "val", "test"]:
    for b in ["train", "val", "test"]:
        if a < b:
            overlap = patients[a] & patients[b]
            print(a, b, len(overlap))


In [ ]:
# Lablel distribution

label_prev = []

for split, df in data.groupby("split"):
    for label in LABEL_COLS:
        n_pos = df[label].sum()
        n_total = len(df)
        label_prev.append({
            "split": split,
            "label": label,
            "positive": int(n_pos),
            "total": int(n_total),
            "prevalence": n_pos / n_total,
        })

label_prev = pd.DataFrame(label_prev)

label_prev.sort_values(["split", "prevalence"], ascending=[True, False])


In [ ]:
# Plot label imbalance

train_prev = (
    label_prev[label_prev["split"] == "train"]
    .sort_values("prevalence", ascending=False)
    .reset_index(drop=True)
)

# Print counts
print(train_prev[["label", "positive", "total", "prevalence"]].to_string(index=False))

# Plot prevalence with counts
plt.figure(figsize=(10, 8))
ax = sns.barplot(data=train_prev, x="prevalence", y="label")

plt.title("Train Label Prevalence")
plt.xlabel("Positive fraction")
plt.ylabel("")

for i, row in train_prev.iterrows():
    ax.text(
        row["prevalence"] + 0.005,
        i,
        f'{int(row["positive"]):,} / {int(row["total"]):,} ({row["prevalence"]:.1%})',
        va="center",
        fontsize=9,
    )

ax.set_xlim(0, min(1.0, train_prev["prevalence"].max() + 0.12))

plt.tight_layout()
plt.show()


In [ ]:
# Check missing

cols_to_check = [
    "clinical_indication",
    "temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp",
    "gender", "ViewPosition", "path",
]

missing = []

for split, df in data.groupby("split"):
    for col in cols_to_check:
        missing.append({
            "split": split,
            "column": col,
            "missing": df[col].isna().sum(),
            "total": len(df),
            "missing_rate": df[col].isna().mean(),
        })

missing = pd.DataFrame(missing)
missing.sort_values(["split", "missing_rate"], ascending=[True, False])


In [ ]:
# View distribution

view_dist = (
    data.groupby(["split", "ViewPosition"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "count"], ascending=[True, False])
)

view_dist


In [ ]:
# Label re appearance

label_matrix = data[LABEL_COLS].astype(int)

cooccurrence = label_matrix.T @ label_matrix
cooccurrence


In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(cooccurrence, cmap="mako")
plt.title("Label Co-occurrence Counts")
plt.tight_layout()
plt.show()


In [ ]:
# Conditional co-occurrence

conditional = cooccurrence.div(np.diag(cooccurrence), axis=0)

plt.figure(figsize=(12, 10))
sns.heatmap(conditional, cmap="inferno", vmin=0, vmax=1)
plt.title("Conditional Label Co-occurrence: P(label B | label A)")
plt.tight_layout()
plt.show()


In [ ]:
# Vital by disease label

vital_summary = []

for label in LABEL_COLS:
    pos = data[data[label] == 1]
    neg = data[data[label] == 0]

    for vital in VITAL_COLS:
        vital_summary.append({
            "label": label,
            "vital": vital,
            "positive_mean": pos[vital].mean(),
            "negative_mean": neg[vital].mean(),
            "difference": pos[vital].mean() - neg[vital].mean(),
            "positive_n": pos[vital].notna().sum(),
            "negative_n": neg[vital].notna().sum(),
        })

vital_summary = pd.DataFrame(vital_summary)

vital_summary.sort_values("difference", key=abs, ascending=False).head(30)


In [ ]:
# Report / Text length

data["clinical_indication_words"] = (
    data["clinical_indication"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

text_summary = (
    data.groupby("split")["clinical_indication_words"]
    .describe()
)

text_summary


In [ ]:
# Multilable density

data["num_positive_labels"] = data[LABEL_COLS].sum(axis=1)

data.groupby("split")["num_positive_labels"].describe()


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=data, x="num_positive_labels", hue="split", discrete=True, multiple="dodge")
plt.title("Number of Positive Labels per Image")
plt.xlabel("Positive labels")
plt.tight_layout()
plt.show()
